# Crowding Robustness Benchmark
ResNet / ViT / ConvKAN karşılaştırması

**Adımlar:**
1. Repo klonla
2. Bağımlılıkları kur
3. COCO indir
4. Veri hazırla
5. Modelleri eğit
6. Değerlendir ve görselleştir

## 0. GPU Kontrolü

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Repo Klonla

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/crowding-robustness.git'  # <-- degistir
REPO_DIR = 'crowding-robustness'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
print('Calisma dizini:', os.getcwd())

## 2. Bağımlılıkları Kur

In [ ]:
import os, sys

# Temel bagimliliklar
os.system('pip install -q -r requirements.txt')
os.system('pip install -q timm fvcore')

# ConvKAN - PyPI'da yok, repo clone edilir
if not os.path.exists('torch-conv-kan'):
    os.system('git clone https://github.com/IvanDrokin/torch-conv-kan.git')
os.system('cd torch-conv-kan && pip install -q -r requirements.txt')
sys.path.insert(0, os.path.abspath('torch-conv-kan'))

# Kontrol
from kan_convs import KANConv2DLayer
print('KANConv2DLayer: OK')
print('Tum kurulumlar tamamlandi.')

## 3. COCO İndir
> Secenek A: Direkt indir (~18GB)
> Secenek B: Google Drive'dan mount et (onerilir)

In [ ]:
import os

# --- Secenek A: Direkt indir ---
# os.system('bash scripts/download_coco.sh')

# --- Secenek B: Google Drive mount (onerilir) ---
from google.colab import drive
drive.mount('/content/drive')

COCO_DRIVE_PATH = '/content/drive/MyDrive/coco'  # <-- kendi yolunu gir
if os.path.exists(COCO_DRIVE_PATH) and not os.path.exists('coco'):
    os.symlink(COCO_DRIVE_PATH, 'coco')
    print('COCO Drive dan baglandi.')
elif os.path.exists('coco'):
    print('COCO zaten mevcut.')
else:
    print('COCO bulunamadi. Secenek A yi deneyin.')

## 4. Veri Hazırla

In [ ]:
import os
# Instance kirpma + split
os.system('python data/prepare_dataset.py --config configs/config.yaml')

In [ ]:
import os
# Crowding kompozitlerini olustur (leakage kontrolu dahil)
os.system('python data/build_composites.py --config configs/config.yaml')

In [ ]:
# Ornek gorsellere bak
import matplotlib.pyplot as plt
import cv2, glob, random

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Ust: Isolation (train) | Alt: Crowded (test, 2deg, same_class)')

iso_paths   = glob.glob('dataset/train/*/*.png')
crowd_paths = glob.glob('dataset/composites/test/*/same_class/2deg/*.png')

for i, ax in enumerate(axes[0]):
    if i < len(iso_paths):
        img = cv2.cvtColor(cv2.imread(random.choice(iso_paths)), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis('off')

for i, ax in enumerate(axes[1]):
    if i < len(crowd_paths):
        img = cv2.cvtColor(cv2.imread(random.choice(crowd_paths)), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis('off')

plt.tight_layout(); plt.show()

## 5. Model Eğitimi

In [ ]:
import os
# Tek model egit
MODEL = 'resnet50'  # resnet34, resnet50, resnet101, vit_s_16, vit_b_16, convkan_s, convkan_m, convkan_l
os.system(f'python train.py --model {MODEL} --config configs/config.yaml')

In [ ]:
import os
# Tum modelleri siraya egit
MODELS = ['resnet34', 'resnet50', 'resnet101', 'vit_s_16', 'vit_b_16', 'convkan_s', 'convkan_m', 'convkan_l']
for m in MODELS:
    print(f'=== {m} egitiliyor ===')
    os.system(f'python train.py --model {m} --config configs/config.yaml')

## 6. Değerlendirme

In [ ]:
import os
os.system('python evaluate.py --config configs/config.yaml')

## 7. Görselleştirme

In [ ]:
import os
# Tum grafikler
os.system('python visualize.py --config configs/config.yaml')

# Grad-CAM da dahil
# os.system('python visualize.py --gradcam --config configs/config.yaml')

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('results/figures')
for fig_path in sorted(fig_dir.glob('*.png')):
    print(f'--- {fig_path.name} ---')
    display(Image(str(fig_path)))

## 8. Sonuçları Drive'a Kaydet

In [ ]:
import shutil
SAVE_PATH = '/content/drive/MyDrive/crowding_results'
shutil.copytree('results', SAVE_PATH, dirs_exist_ok=True)
print(f'Sonuclar kaydedildi: {SAVE_PATH}')